# 01. EDA — KuaiRec & KuaiRand 탐색적 데이터 분석

## 프로젝트 개요
- **목표**: 숏폼 영상 플랫폼(Kuaishou)의 추천 알고리즘 A/B 테스트 분석
- **KuaiRec**: 1,411 users × 3,327 videos — 99.6% 밀도 완전관측 행렬
- **KuaiRand-Pure**: 27K users — `is_rand` 플래그로 랜덤 vs 추천 노출 구분, 피드백 12종

---

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

DATA_DIR = Path('../data')
print("Setup complete!")

## 1. KuaiRand — 랜덤 vs 추천 노출 데이터

In [ ]:
# 랜덤 노출 로그 (Control)
log_rand = pd.read_csv(DATA_DIR / 'kuairand/KuaiRand-Pure/data/log_random_4_22_to_5_08_pure.csv')
# 추천 노출 로그 (Treatment) — 같은 기간
log_std_same = pd.read_csv(DATA_DIR / 'kuairand/KuaiRand-Pure/data/log_standard_4_22_to_5_08_pure.csv')
# 추천 노출 로그 — 이전 기간
log_std_prev = pd.read_csv(DATA_DIR / 'kuairand/KuaiRand-Pure/data/log_standard_4_08_to_4_21_pure.csv')

print(f"랜덤 노출 로그: {log_rand.shape[0]:,} rows")
print(f"추천 노출 로그 (같은 기간): {log_std_same.shape[0]:,} rows")
print(f"추천 노출 로그 (이전 기간): {log_std_prev.shape[0]:,} rows")
print(f"\n컬럼: {list(log_rand.columns)}")

In [ ]:
# 기본 통계
print("=== 랜덤 노출 로그 기본 정보 ===")
print(f"유저 수: {log_rand['user_id'].nunique():,}")
print(f"비디오 수: {log_rand['video_id'].nunique():,}")
print(f"날짜 범위: {log_rand['date'].min()} ~ {log_rand['date'].max()}")
print(f"\nis_rand 분포:\n{log_rand['is_rand'].value_counts()}")
print(f"\n=== 추천 노출 로그 (같은 기간) ===")
print(f"유저 수: {log_std_same['user_id'].nunique():,}")
print(f"비디오 수: {log_std_same['video_id'].nunique():,}")
print(f"is_rand 분포:\n{log_std_same['is_rand'].value_counts()}")

### 1.1 피드백 신호 비교: 랜덤 노출 vs 추천 노출

In [ ]:
# 같은 기간(4/22~5/08) 데이터로 비교 — 공정한 A/B 비교
# 랜덤 로그에서 is_rand=1인 것만 추출 (Control)
control = log_rand[log_rand['is_rand'] == 1].copy()
# 추천 로그에서 is_rand=0인 것만 추출 (Treatment)
treatment = log_std_same[log_std_same['is_rand'] == 0].copy()

feedback_cols = ['is_click', 'is_like', 'is_follow', 'is_comment', 'is_forward', 'is_hate', 'long_view']

# 각 피드백 신호의 비율 계산
comparison = pd.DataFrame({
    'Random (Control)': control[feedback_cols].mean(),
    'Recommended (Treatment)': treatment[feedback_cols].mean(),
})
comparison['Lift (%)'] = ((comparison['Recommended (Treatment)'] - comparison['Random (Control)']) 
                          / comparison['Random (Control)'] * 100)
comparison['Abs Diff'] = comparison['Recommended (Treatment)'] - comparison['Random (Control)']

print(f"Control (랜덤 노출): {len(control):,} rows")
print(f"Treatment (추천 노출): {len(treatment):,} rows\n")
comparison.round(4)

In [ ]:
# 피드백 비율 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 왼쪽: 비율 비교 바 차트
x = np.arange(len(feedback_cols))
width = 0.35
bars1 = axes[0].bar(x - width/2, comparison['Random (Control)'], width, label='Random (Control)', color='#3498db', alpha=0.8)
bars2 = axes[0].bar(x + width/2, comparison['Recommended (Treatment)'], width, label='Recommended (Treatment)', color='#e74c3c', alpha=0.8)
axes[0].set_ylabel('Rate')
axes[0].set_title('Feedback Signal Rates: Random vs Recommended')
axes[0].set_xticks(x)
axes[0].set_xticklabels(feedback_cols, rotation=45, ha='right')
axes[0].legend()

# 오른쪽: Lift (%) 바 차트
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in comparison['Lift (%)']]
axes[1].bar(feedback_cols, comparison['Lift (%)'], color=colors, alpha=0.8)
axes[1].set_ylabel('Lift (%)')
axes[1].set_title('Recommendation Lift over Random Exposure')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_xticklabels(feedback_cols, rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../notebooks/figures/01_feedback_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.2 시청 시간 분포 비교

In [ ]:
# 시청 시간 분포 비교 (이상치 제거: 상위 1%)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# play_time_ms 분포
cap = control['play_time_ms'].quantile(0.99)
axes[0].hist(control['play_time_ms'].clip(upper=cap), bins=50, alpha=0.6, label='Random', color='#3498db', density=True)
axes[0].hist(treatment['play_time_ms'].clip(upper=cap), bins=50, alpha=0.6, label='Recommended', color='#e74c3c', density=True)
axes[0].set_xlabel('Play Time (ms)')
axes[0].set_ylabel('Density')
axes[0].set_title('Play Time Distribution')
axes[0].legend()

# watch_ratio = play_time / duration
control['watch_ratio'] = control['play_time_ms'] / control['duration_ms'].replace(0, np.nan)
treatment['watch_ratio'] = treatment['play_time_ms'] / treatment['duration_ms'].replace(0, np.nan)

wr_cap = 3.0  # 3배 이상 반복 시청은 클리핑
axes[1].hist(control['watch_ratio'].clip(upper=wr_cap).dropna(), bins=50, alpha=0.6, label='Random', color='#3498db', density=True)
axes[1].hist(treatment['watch_ratio'].clip(upper=wr_cap).dropna(), bins=50, alpha=0.6, label='Recommended', color='#e74c3c', density=True)
axes[1].set_xlabel('Watch Ratio (play_time / duration)')
axes[1].set_ylabel('Density')
axes[1].set_title('Watch Ratio Distribution')
axes[1].axvline(x=1.0, color='black', linestyle='--', alpha=0.5, label='Full Watch')
axes[1].legend()

plt.tight_layout()
plt.savefig('../notebooks/figures/01_playtime_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n평균 시청 시간 (ms):")
print(f"  Random:      {control['play_time_ms'].mean():,.0f}")
print(f"  Recommended: {treatment['play_time_ms'].mean():,.0f}")
print(f"\n평균 Watch Ratio:")
print(f"  Random:      {control['watch_ratio'].mean():.3f}")
print(f"  Recommended: {treatment['watch_ratio'].mean():.3f}")

### 1.3 유저 특성 분석

In [ ]:
# 유저 특성 로드
user_feat = pd.read_csv(DATA_DIR / 'kuairand/KuaiRand-Pure/data/user_features_pure.csv')
print(f"유저 수: {len(user_feat):,}")
print(f"\n컬럼: {list(user_feat.columns)}")
user_feat.head()

In [ ]:
# 유저 활동 수준 분포
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 유저 활동도 분포
active_counts = user_feat['user_active_degree'].value_counts()
axes[0].bar(active_counts.index, active_counts.values, color='#3498db', alpha=0.8)
axes[0].set_title('User Active Degree Distribution')
axes[0].set_xlabel('Active Degree')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# 팔로워 수 분포
axes[1].bar(user_feat['fans_user_num_range'].value_counts().index, 
            user_feat['fans_user_num_range'].value_counts().values, color='#e74c3c', alpha=0.8)
axes[1].set_title('Fans Count Range Distribution')
axes[1].set_xlabel('Fans Range')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

# 가입 일수 분포
axes[2].bar(user_feat['register_days_range'].value_counts().index,
            user_feat['register_days_range'].value_counts().values, color='#2ecc71', alpha=0.8)
axes[2].set_title('Registration Days Range Distribution')
axes[2].set_xlabel('Days Range')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../notebooks/figures/01_user_features.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.4 유저 활동도별 피드백 차이 (랜덤 vs 추천)

In [ ]:
# 유저 활동도와 로그 데이터 조인
control_w_feat = control.merge(user_feat[['user_id', 'user_active_degree']], on='user_id', how='left')
treatment_w_feat = treatment.merge(user_feat[['user_id', 'user_active_degree']], on='user_id', how='left')

# 활동도별 주요 지표 비교
metrics = ['is_click', 'is_like', 'long_view']
active_degrees = ['full_active', 'high_active', 'middle_active', 'low_active']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, metric in enumerate(metrics):
    ctrl_vals = []
    treat_vals = []
    labels = []
    for degree in active_degrees:
        c = control_w_feat[control_w_feat['user_active_degree'] == degree][metric].mean()
        t = treatment_w_feat[treatment_w_feat['user_active_degree'] == degree][metric].mean()
        if not np.isnan(c) and not np.isnan(t):
            ctrl_vals.append(c)
            treat_vals.append(t)
            labels.append(degree.replace('_active', ''))
    
    x = np.arange(len(labels))
    axes[i].bar(x - 0.2, ctrl_vals, 0.4, label='Random', color='#3498db', alpha=0.8)
    axes[i].bar(x + 0.2, treat_vals, 0.4, label='Recommended', color='#e74c3c', alpha=0.8)
    axes[i].set_title(f'{metric} by User Activity')
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(labels, rotation=30)
    axes[i].legend()

plt.tight_layout()
plt.savefig('../notebooks/figures/01_activity_feedback.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.5 일별 추세 분석

In [ ]:
# 일별 CTR 추이 비교
ctrl_daily = control.groupby('date').agg(
    ctr=('is_click', 'mean'),
    like_rate=('is_like', 'mean'),
    n=('is_click', 'count')
).reset_index()
treat_daily = treatment.groupby('date').agg(
    ctr=('is_click', 'mean'),
    like_rate=('is_like', 'mean'),
    n=('is_click', 'count')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# CTR 추이
axes[0].plot(ctrl_daily['date'].astype(str), ctrl_daily['ctr'], 'o-', label='Random', color='#3498db')
axes[0].plot(treat_daily['date'].astype(str), treat_daily['ctr'], 's-', label='Recommended', color='#e74c3c')
axes[0].set_title('Daily CTR Trend')
axes[0].set_ylabel('CTR')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# 일별 노출 수
axes[1].bar(ctrl_daily['date'].astype(str), ctrl_daily['n'], alpha=0.6, label='Random', color='#3498db')
axes[1].bar(treat_daily['date'].astype(str), treat_daily['n'], alpha=0.6, label='Recommended', color='#e74c3c')
axes[1].set_title('Daily Impression Count')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../notebooks/figures/01_daily_trend.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. KuaiRec — 완전관측 행렬 탐색

In [ ]:
# KuaiRec 소규모 완전관측 행렬
small_mat = pd.read_csv(DATA_DIR / 'kuairec/KuaiRec 2.0/data/small_matrix.csv')
print(f"Small Matrix: {small_mat.shape[0]:,} rows")
print(f"유저 수: {small_mat['user_id'].nunique():,}")
print(f"비디오 수: {small_mat['video_id'].nunique():,}")
print(f"\n밀도: {small_mat.shape[0] / (small_mat['user_id'].nunique() * small_mat['video_id'].nunique()) * 100:.1f}%")
print(f"\n컬럼: {list(small_mat.columns)}")
small_mat.head()

In [ ]:
# watch_ratio 분포 — 완전관측 행렬의 핵심 지표
small_mat['watch_ratio_clipped'] = small_mat['watch_ratio'].clip(upper=5)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# watch_ratio 히스토그램
axes[0].hist(small_mat['watch_ratio_clipped'], bins=100, color='#9b59b6', alpha=0.7, edgecolor='white')
axes[0].set_title('Watch Ratio Distribution (KuaiRec)')
axes[0].set_xlabel('Watch Ratio')
axes[0].set_ylabel('Count')
axes[0].axvline(x=1.0, color='red', linestyle='--', label='Full Watch')
axes[0].legend()

# 유저별 평균 watch_ratio 분포
user_avg_wr = small_mat.groupby('user_id')['watch_ratio'].mean()
axes[1].hist(user_avg_wr, bins=50, color='#3498db', alpha=0.7, edgecolor='white')
axes[1].set_title('Per-User Avg Watch Ratio')
axes[1].set_xlabel('Avg Watch Ratio')
axes[1].set_ylabel('User Count')

# 비디오별 평균 watch_ratio 분포
video_avg_wr = small_mat.groupby('video_id')['watch_ratio'].mean()
axes[2].hist(video_avg_wr, bins=50, color='#e74c3c', alpha=0.7, edgecolor='white')
axes[2].set_title('Per-Video Avg Watch Ratio')
axes[2].set_xlabel('Avg Watch Ratio')
axes[2].set_ylabel('Video Count')

plt.tight_layout()
plt.savefig('../notebooks/figures/01_kuairec_watch_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.1 유저-아이템 상호작용 히트맵 (완전관측 행렬)

In [ ]:
# 상위 50 유저 x 상위 50 비디오 히트맵 (시각화용 샘플)
top_users = small_mat.groupby('user_id')['watch_ratio'].mean().nlargest(50).index
top_videos = small_mat.groupby('video_id')['watch_ratio'].mean().nlargest(50).index

sub = small_mat[small_mat['user_id'].isin(top_users) & small_mat['video_id'].isin(top_videos)]
pivot = sub.pivot_table(index='user_id', columns='video_id', values='watch_ratio', aggfunc='mean')

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(pivot, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Watch Ratio'})
ax.set_title('User-Item Interaction Heatmap (Top 50 Users × Top 50 Videos)')
ax.set_xlabel('Video ID')
ax.set_ylabel('User ID')
plt.tight_layout()
plt.savefig('../notebooks/figures/01_interaction_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 KuaiRec 아이템 카테고리 분석

In [ ]:
# 아이템 카테고리 로드
item_cat = pd.read_csv(DATA_DIR / 'kuairec/KuaiRec 2.0/data/item_categories.csv')
print(f"아이템 수: {len(item_cat):,}")
print(f"컬럼: {list(item_cat.columns)}")
item_cat.head(10)

---
## 3. 비디오 특성 분석 (KuaiRand)

In [ ]:
# 비디오 특성 로드
video_basic = pd.read_csv(DATA_DIR / 'kuairand/KuaiRand-Pure/data/video_features_basic_pure.csv')
video_stat = pd.read_csv(DATA_DIR / 'kuairand/KuaiRand-Pure/data/video_features_statistic_pure.csv')

print(f"비디오 기본 특성: {video_basic.shape}")
print(f"컬럼: {list(video_basic.columns)}\n")
print(f"비디오 통계 특성: {video_stat.shape}")
print(f"컬럼: {list(video_stat.columns)[:15]}...")
video_basic.head()

---
## 4. 상관관계 히트맵 — 피드백 신호 간 관계

In [ ]:
# 피드백 신호 간 상관관계
all_feedback_cols = ['is_click', 'is_like', 'is_follow', 'is_comment', 'is_forward', 
                     'is_hate', 'long_view', 'play_time_ms', 'duration_ms']

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Control (랜덤 노출)
corr_ctrl = control[all_feedback_cols].corr()
sns.heatmap(corr_ctrl, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[0], 
            square=True, linewidths=0.5)
axes[0].set_title('Feedback Correlation — Random Exposure')

# Treatment (추천 노출)
corr_treat = treatment[all_feedback_cols].corr()
sns.heatmap(corr_treat, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[1],
            square=True, linewidths=0.5)
axes[1].set_title('Feedback Correlation — Recommended Exposure')

plt.tight_layout()
plt.savefig('../notebooks/figures/01_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. EDA 요약

### Key Findings
1. **데이터 규모**: KuaiRand에 랜덤 노출 ~118만건 + 추천 노출 ~114만건 (같은 기간)
2. **피드백 차이**: 추천 노출이 랜덤 대비 CTR, 좋아요율, 시청완료율에서 얼마나 높은지 확인
3. **유저 세그먼트**: 활동도(full/high/middle/low)에 따라 추천 효과가 다를 수 있음
4. **KuaiRec 완전관측**: 99.6% 밀도 — 추천 알고리즘의 ground-truth 성능 평가 가능
5. **상관관계**: 랜덤/추천 노출에서 피드백 신호 간 상관 구조 차이 존재 여부

### Next Steps
- `02_ab_test_basic.ipynb`: 통계적 가설 검정으로 A/B 효과 유의성 확인
- `03_causal_inference.ipynb`: PSM + CATE로 인과적 추천 효과 분석